In [0]:
1.from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StringType
from delta.tables import DeltaTable


# ============================================================
# 1. CONFIGURATION
# ============================================================

STORAGE_ACCOUNT = "nestlesalesdatalake"
CONTAINER = "customer-sales-datalake"

BASE_PATH = (
    f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
)

TRANSFORM_CONFIG_PATH = (
    BASE_PATH + "metadata/silver_column_transform_config/"
)

SCHEMA_CONFIG_PATH = (
    BASE_PATH + "metadata/schema_config/"
)

SUPPORTED_TRANSFORMATIONS = {
    "TRIM",
    "UPPERCASE",
    "LOWERCASE",
    "NULL_IF_EMPTY",
    "RENAME"
}

STRING_TRANSFORMATIONS = {
    "TRIM",
    "UPPERCASE",
    "LOWERCASE",
    "NULL_IF_EMPTY"
}

In [0]:
# ============================================================
# 2. DATASET PARAMETER
# ============================================================

dbutils.widgets.text(
    "dataset_name",
    "google_ads_campaigns",
    "DATASET NAME"
)

dataset_name = dbutils.widgets.get("dataset_name").strip()

if not dataset_name:
    raise ValueError("dataset_name cannot be empty.")

print(f"Loading transformation configuration for: {dataset_name}")

Loading transformation configuration for: google_ads_campaigns


In [0]:
# ============================================================
# 3. LOAD ACTIVE SCHEMA CONFIGURATION
# ============================================================

if not DeltaTable.isDeltaTable(spark, SCHEMA_CONFIG_PATH):
    raise RuntimeError(
        f"Schema configuration Delta table not found: "
        f"{SCHEMA_CONFIG_PATH}"
    )

schema_config_df = (
    spark.read
    .format("delta")
    .load(SCHEMA_CONFIG_PATH)
)

active_schema_rows = (
    schema_config_df
    .filter(
        (F.lower(F.trim(F.col("dataset_name"))) == dataset_name.lower()) &
        (F.col("is_active") == True)
    )
    .select(
        "schema_ddl",
        "schema_version",
        "source_system"
    )
    .collect()
)

if len(active_schema_rows) != 1:
    raise ValueError(
        f"Expected exactly one active schema configuration "
        f"for {dataset_name}, found {len(active_schema_rows)}"
    )

schema_row = active_schema_rows[0]

schema_ddl = schema_row["schema_ddl"]
schema_version = schema_row["schema_version"]
source_system = schema_row["source_system"]

if not schema_ddl:
    raise ValueError(
        f"schema_ddl is empty for dataset {dataset_name}"
    )

schema_struct = StructType.fromDDL(schema_ddl)

# Dictionary containing column names and their Spark data types.
available_fields = {
    field.name.lower(): field.dataType
    for field in schema_struct.fields
}

available_columns = set(available_fields.keys())

print(f"Source system       : {source_system}")
print(f"Active schema version: {schema_version}")
print(f"Schema columns found : {len(available_columns)}")

Source system       : google_ads
Active schema version: 1
Schema columns found : 18


In [0]:
# ============================================================
# 4. LOAD ACTIVE TRANSFORMATION CONFIGURATION
# ============================================================

if not DeltaTable.isDeltaTable(spark, TRANSFORM_CONFIG_PATH):
    raise RuntimeError(
        f"Transformation configuration Delta table not found: "
        f"{TRANSFORM_CONFIG_PATH}"
    )

transformation_config_df = (
    spark.read
    .format("delta")
    .load(TRANSFORM_CONFIG_PATH)
)

dataset_rules_df = (
    transformation_config_df
    .filter(
        (F.lower(F.trim(F.col("dataset_name"))) == dataset_name.lower()) &
        (F.col("is_active") == True)
    )
    .orderBy(
        F.col("execution_order").asc(),
        F.col("source_column").asc()
    )
)

rules = dataset_rules_df.collect()

print(f"Active transformation rules found: {len(rules)}")

display(dataset_rules_df)

Active transformation rules found: 8


dataset_name,source_column,target_column,transformation_type,transformation_parameters,execution_order,is_active,description,created_at,updated_at
google_ads_campaigns,business_brand,business_brand,TRIM,null,1,true,TRIM applied to business_brand,2026-10-31T13:18:26.059Z,2026-10-31T13:29:05.905Z
google_ads_campaigns,business_objective,business_objective,TRIM,null,1,true,TRIM applied to business_objective,2026-10-31T13:18:26.059Z,2026-10-31T13:29:05.905Z
google_ads_campaigns,campaign_name,campaign_name,TRIM,null,1,true,TRIM applied to campaign_name,2026-10-31T13:18:26.059Z,2026-10-31T13:29:05.905Z
google_ads_campaigns,product_category,product_category,TRIM,null,1,true,TRIM applied to product_category,2026-10-31T13:18:26.059Z,2026-10-31T13:29:05.905Z
google_ads_campaigns,business_brand,business_brand,NULL_IF_EMPTY,null,2,true,NULL_IF_EMPTY applied to business_brand,2026-10-31T13:18:26.059Z,2026-10-31T13:29:05.905Z
google_ads_campaigns,business_objective,business_objective,NULL_IF_EMPTY,null,2,true,NULL_IF_EMPTY applied to business_objective,2026-10-31T13:18:26.059Z,2026-10-31T13:29:05.905Z
google_ads_campaigns,campaign_name,campaign_name,NULL_IF_EMPTY,null,2,true,NULL_IF_EMPTY applied to campaign_name,2026-10-31T13:18:26.059Z,2026-10-31T13:29:05.905Z
google_ads_campaigns,product_category,product_category,NULL_IF_EMPTY,null,2,true,NULL_IF_EMPTY applied to product_category,2026-10-31T13:18:26.059Z,2026-10-31T13:29:05.905Z


In [0]:

# ============================================================
# 5. VALIDATE TRANSFORMATION CONFIGURATION
# ============================================================

validation_errors = []
seen_rule_keys = set()

for rule in rules:

    source_column = (
        rule["source_column"].strip()
        if rule["source_column"]
        else ""
    )

    target_column = (
        rule["target_column"].strip()
        if rule["target_column"]
        else ""
    )

    transformation_type = (
        rule["transformation_type"].strip().upper()
        if rule["transformation_type"]
        else ""
    )

    execution_order = rule["execution_order"]

    # --------------------------------------------------------
    # 5.1 Validate transformation type
    # --------------------------------------------------------

    if transformation_type not in SUPPORTED_TRANSFORMATIONS:
        validation_errors.append(
            f"Unsupported transformation type "
            f"'{transformation_type}' for column '{source_column}'."
        )

    # --------------------------------------------------------
    # 5.2 Validate source column
    # --------------------------------------------------------

    if not source_column:
        validation_errors.append(
            "A transformation rule has an empty source_column."
        )

    elif source_column.lower() not in available_columns:
        validation_errors.append(
            f"Source column '{source_column}' does not exist "
            f"in the active schema."
        )

    # --------------------------------------------------------
    # 5.3 Validate target column
    # --------------------------------------------------------

    if not target_column:
        validation_errors.append(
            f"Transformation rule for '{source_column}' "
            f"must specify target_column."
        )

    # --------------------------------------------------------
    # 5.4 Validate string transformation data types
    # --------------------------------------------------------

    if (
        transformation_type in STRING_TRANSFORMATIONS
        and source_column.lower() in available_fields
    ):

        source_data_type = available_fields[source_column.lower()]

        if not isinstance(source_data_type, StringType):
            validation_errors.append(
                f"{transformation_type} requires a string column, "
                f"but '{source_column}' has type "
                f"'{source_data_type}'."
            )

    # --------------------------------------------------------
    # 5.5 Validate RENAME target
    # --------------------------------------------------------

    if (
        transformation_type == "RENAME"
        and source_column
        and target_column
    ):

        if (
            target_column.lower() in available_columns
            and target_column.lower() != source_column.lower()
        ):
            validation_errors.append(
                f"RENAME target '{target_column}' already exists "
                f"in the active schema. Check for a naming conflict."
            )

    # --------------------------------------------------------
    # 5.6 Validate execution order
    # --------------------------------------------------------

    if execution_order is None:
        validation_errors.append(
            f"Missing execution_order for '{source_column}'."
        )

    elif execution_order < 1:
        validation_errors.append(
            f"Invalid execution_order for '{source_column}'. "
            f"Execution order must be greater than or equal to 1."
        )

    # --------------------------------------------------------
    # 5.7 Validate duplicate transformation rules
    # --------------------------------------------------------

    rule_key = (
        source_column.lower(),
        transformation_type
    )

    if rule_key in seen_rule_keys:
        validation_errors.append(
            f"Duplicate transformation rule found for "
            f"'{source_column}' and '{transformation_type}'."
        )

    seen_rule_keys.add(rule_key)

    # --------------------------------------------------------
    # 5.8 Validate transformation parameters representation
    # --------------------------------------------------------

    parameters = rule["transformation_parameters"]

    if parameters is not None and not isinstance(parameters, str):
        validation_errors.append(
            f"Invalid transformation_parameters for "
            f"'{source_column}'. Expected a string or NULL."
        )


# Stop execution if validation errors exist.

if validation_errors:

    print("Transformation configuration validation failed:")

    for error in validation_errors:
        print(f"- {error}")

    raise ValueError(
        "Fix the transformation metadata before continuing."
    )


print("All transformation configurations passed validation.")


All transformation configurations passed validation.


In [0]:
# ============================================================
# 6. BUILD REUSABLE CONFIGURATION STRUCTURE
# ============================================================

transformation_rules = []

for rule in rules:

    transformation_rules.append({

        "dataset_name": rule["dataset_name"],

        "source_column": rule["source_column"],

        "target_column": rule["target_column"],

        "transformation_type": (
            rule["transformation_type"].strip().upper()
        ),

        # transformation_parameters is StringType in Notebook 02.
        # Therefore, preserve it as a string instead of converting
        # it into a dictionary.
        "transformation_parameters": (
            rule["transformation_parameters"]
        ),

        "execution_order": rule["execution_order"],

        "description": rule["description"]

    })

In [0]:
# ============================================================
# 7. DISPLAY LOADED CONFIGURATION
# ============================================================

print("================================================")
print("TRANSFORMATION CONFIGURATION LOADED")
print("================================================")

print(f"Dataset        : {dataset_name}")
print(f"Source system  : {source_system}")
print(f"Schema version : {schema_version}")
print(f"Active rules   : {len(transformation_rules)}")

if transformation_rules:

    for rule in transformation_rules:
        print(rule)

else:
    print("No active transformation rules configured.")

print("================================================")
print("Configuration loading and validation completed.")

TRANSFORMATION CONFIGURATION LOADED
Dataset        : google_ads_campaigns
Source system  : google_ads
Schema version : 1
Active rules   : 8
{'dataset_name': 'google_ads_campaigns', 'source_column': 'business_brand', 'target_column': 'business_brand', 'transformation_type': 'TRIM', 'transformation_parameters': None, 'execution_order': 1, 'description': 'TRIM applied to business_brand'}
{'dataset_name': 'google_ads_campaigns', 'source_column': 'business_objective', 'target_column': 'business_objective', 'transformation_type': 'TRIM', 'transformation_parameters': None, 'execution_order': 1, 'description': 'TRIM applied to business_objective'}
{'dataset_name': 'google_ads_campaigns', 'source_column': 'campaign_name', 'target_column': 'campaign_name', 'transformation_type': 'TRIM', 'transformation_parameters': None, 'execution_order': 1, 'description': 'TRIM applied to campaign_name'}
{'dataset_name': 'google_ads_campaigns', 'source_column': 'product_category', 'target_column': 'product_cat